# 09 · Inference Deployment —— 图书馆分格，学徒起草，压缩包上架

**家族位置**：08 生产级收官站。01~08 把位置/算力/参数/知识全优化完，本章管上线：PagedAttention（KV 分页防碎片）+ 投机解码（小模型打草稿）+ GGUF（量化格式上架）。

**学习目标**：页式 KV 管理（分配/碎片/释放）；draft+target 并行验证（接受率）；INT8 文件拼盘体积比；三件套可叠加。

## 1. 原理：分格书架 + 学徒起草 + 真空压缩包

### 通俗理解

**一句话**：PagedAttention 像书架分格——纪要本按页租格，长短会议混排不浪费；投机解码像学徒起草、师傅改——草稿对了整段留，错了只改错字；GGUF 像真空压缩包——INT8 抽气+贴签（魔数/头），CPU 直接拆包读。

### 结构账

```
分页： PagedKVManager(block=8)：seq 按 8 切页，按需租/释放回池；口径=页数+内碎片率
投机： draft(GQA kv=1) 猜 γ=4，target(MHA kv=4) 并行验证；口径=接受率+有效token/轮
GGUF： toy 拼盘（魔数+头+scale+INT8 体）：体积比≈4；回读校验 max|Δ| 口径
基座： 02 站 GQADecoder（copy 混合短训，保证 draft/target 行为可比）
```

In [ ]:
import sys, copy
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, ConcatDataset
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import make_copy_data
from common.models import GQADecoder
from common.infer import PagedKVManager, speculative_decode, gguf_bytes
from common.utils import set_seed,setup_chinese_font,count_params
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
Xc,yc=make_copy_data(2000,16,16,seed=0); Xv,yv=make_copy_data(300,16,16,seed=1)
tr=DataLoader(TensorDataset(Xc,yc),batch_size=128,shuffle=True)
va=DataLoader(TensorDataset(Xv,yv),batch_size=512)
torch.manual_seed(0)
target=GQADecoder(vocab=16,dim=64,depth=2,q_heads=4,kv_heads=4)
torch.manual_seed(0)
draft=GQADecoder(vocab=16,dim=64,depth=2,q_heads=4,kv_heads=1)
import torch.nn as nn
def fit_short(m,epochs=8):
    opt=torch.optim.Adam(m.parameters(),lr=3e-3); crit=nn.CrossEntropyLoss()
    for ep in range(1,epochs+1):
        m.train(); tot=0
        for src,tgt in tr:
            loss=crit(m(src).reshape(-1,16),tgt.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()*len(src)
        print(f'ep {ep} loss {tot/len(tr.dataset):.3f}',flush=True)
fit_short(target); fit_short(draft)
print(f'target {count_params(target)} / draft {count_params(draft)}（同起点同数据，行为可比）',flush=True)

## 2. PagedAttention：变长混排，碎片率实测

In [ ]:
mgr=PagedKVManager(block_size=8,max_blocks=64)
for sid,L in [('A',8),('B',20),('C',33)]:
    mgr.new_seq(sid); mgr.append_tokens(sid,L)
rows={sid:mgr.usage(sid) for sid in ['A','B','C']}
for sid,r in rows.items(): print(f'seq{sid}: 页={r["pages"]} tokens={r["tokens"]} 碎片={r["waste"]} ({r["waste_rate"]:.1%})',flush=True)
mgr.free_seq('B'); print('释放B后空闲页:',len(mgr.free),flush=True)
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(list(rows),[r['pages'] for r in rows.values()],color=['#4C72B0','#55A868','#DD8452'])
for i,r in enumerate(rows.values()): ax.text(i,r['pages']+0.05,f"{r['pages']}页 碎片{r['waste_rate']:.0%}",ha='center')
ax.set_title('页式分配：8/20/33 tokens → 1/3/5 页'); ax.set_ylabel('pages')
plt.tight_layout(); plt.savefig(FIGS/'fig1_paged.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 投机解码：draft 猜 4 个，target 一次验

In [ ]:
torch.manual_seed(7)
pre=torch.randint(0,16,(2,8))
ids,acc,rounds=speculative_decode(draft,target,pre,gamma=4,max_new=16)
ref=target.generate(pre,ids.size(1)-pre.size(1),use_cache=False)
print(f'接受={acc} tokens / {rounds} 轮 = {acc/rounds:.2f} token/轮；与 target 直推一致={bool((ids==ref).all())}',flush=True)
fig,ax=plt.subplots(figsize=(6,2.8)); ax.axis('off')
ax.text(0.02,0.7,f'接受率口径：{acc} tokens / {rounds} 轮 = {acc/rounds:.2f} 有效 token/轮（>1 即赚）',fontsize=11)
ax.text(0.02,0.4,'一致性：投机结果 == target 自回归直推（验证器即真值）',fontsize=11)
ax.text(0.02,0.1,'省的是 target 前向次数：1 轮 1 次 vs 自回归 16 次',fontsize=11,color='#1a6b3c')
ax.set_title('投机解码：草稿对了整段留')
plt.tight_layout(); plt.savefig(FIGS/'fig2_spec.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. GGUF：INT8 拼盘体积比 + 回读校验

In [ ]:
g=gguf_bytes(target.state_dict(),bits=8)
print(f'fp32={g["fp32_bytes"]/1e6:.2f}MB → GGUF-toy={g["nbytes"]/1e6:.2f}MB（×{g["ratio"]:.2f}）魔数={g["bytes"][:4]}',flush=True)
from common.models import quantize_int8_per_tensor, dequantize_state
import copy as _copy
qi=_copy.deepcopy(target); qi.load_state_dict(dequantize_state(quantize_int8_per_tensor(target)))
with torch.no_grad():
    errs=[]
    for s,t in va:
        errs.append((target(s)-qi(s)).abs().max().item())
print(f'回读校验 max|Δlogits|={max(errs):.2e}（PTQ 精度口径，见 08-06）',flush=True)
fig,ax=plt.subplots(1,2,figsize=(9,3.2))
ax[0].bar(['fp32','GGUF-toy INT8'],[g['fp32_bytes']/1e6,g['nbytes']/1e6],color=['#4C72B0','#55A868'])
for i,v in enumerate([g['fp32_bytes']/1e6,g['nbytes']/1e6]): ax[0].text(i,v+0.01,f'{v:.2f}MB',ha='center')
ax[0].set_title('上架体积：÷4'); ax[0].set_ylabel('MB')
ax[1].axis('off')
ax[1].text(0.02,0.7,f'三件套叠加：分页（02 碎片）+ 投机（本轮 {acc/rounds:.2f} tok/轮）+ GGUF（×{g["ratio"]:.1f}）',fontsize=10)
ax[1].text(0.02,0.4,'llama.cpp 路径：同格式 CPU 加载（本机拼字节+回读校验，不调外部二进制）',fontsize=10)
ax[1].text(0.02,0.1,'vLLM 不装：Windows 不兼容，PagedAttention 以 toy 模拟为准（家族口径）',fontsize=10,color='#1a6b3c')
ax[1].set_title('08-09 总览')
plt.tight_layout(); plt.savefig(FIGS/'fig3_gguf.png',dpi=150,bbox_inches='tight'); plt.show()
print('SUMMARY',rows,{k:(round(v['waste_rate'],4)) for k,v in rows.items()},round(acc/rounds,4),bool((ids==ref).all()),round(g['ratio'],2),f"{max(errs):.2e}")